# Week 61, Day 5: Managing resources safely

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w061d5.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
import tempfile, os

with tempfile.TemporaryDirectory() as folder:
    path = os.path.join(folder, "scratch.txt")
    with open(path, "w") as f:
        f.write("draft")
    print(os.path.exists(path))
print(os.path.exists(folder))

In [ ]:
import tempfile, os

with tempfile.NamedTemporaryFile("w+", suffix=".csv") as f:
    f.write("name,score\nada,9\n")
    f.flush()
    f.seek(0)
    print(f.read().splitlines(), f.name.endswith(".csv"))
    name = f.name
print(os.path.exists(name))

In [ ]:
class Channel:
    def __init__(self, name):
        self.name, self._closed, self.closes = name, False, 0

    @property
    def closed(self):
        return self._closed

    def send(self, text):
        if self._closed:
            raise ValueError("send on a closed channel")
        return f"{self.name} <- {text}"

    def close(self):
        if not self._closed:
            self._closed = True
            self.closes += 1            # the real work happens only once

    def __enter__(self):
        return self

    def __exit__(self, *exc):
        self.close()
        return False

with Channel("alerts") as ch:
    print(ch.send("disk 90% full"))
ch.close()
print(ch.closed, ch.closes)
try:
    ch.send("again")
except ValueError as e:
    print(e)

In [ ]:
from contextlib import contextmanager

class Pool:
    def __init__(self, factory, size):
        self._free = [factory() for _ in range(size)]

    @contextmanager
    def lease(self):
        if not self._free:
            raise RuntimeError("pool exhausted")
        item = self._free.pop()
        try:
            yield item
        finally:
            self._free.append(item)          # always comes back

    def available(self):
        return len(self._free)

made = iter(range(100))
pool = Pool(lambda: f"conn-{next(made)}", 2)
with pool.lease() as a:
    with pool.lease() as b:
        print(a, b, pool.available())
print(pool.available())

> **Key idea.** Don't use __del__ for cleanup: when it runs isn't guaranteed and its errors are ignored; release explicitly with with, close() and ExitStack. tempfile's TemporaryDirectory and NamedTemporaryFile give scratch space that removes itself. Make close() idempotent, refuse use after closing with ValueError, and offer a closed property and a context manager. A pool hands out a fixed number of resources through a context manager that always takes them back, replacing ones that broke. Deadlines are cooperative: a job checks a deadline between steps and the TimeoutError unwinds its with blocks. In tests, count acquisitions and releases to catch leaks.

### Exercise: Scratch space

Write `in_scratch(func)`: create a `tempfile.TemporaryDirectory`, call `func(folder)` and return its result; the folder must be gone afterwards, even if `func` raises. Then `scratch_file_lines(lines)`: write the lines to a `NamedTemporaryFile` (mode `"w+"`), read them back from the start, and return them, with the file deleted at the end.

In [ ]:
import tempfile, os

def in_scratch(func):
    ___

def scratch_file_lines(lines):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import os\ndef test_scratch():\n    \"tempfile\"\n    seen = []\n    def job(folder):\n        seen.append(folder)\n        with open(os.path.join(folder, \"x.txt\"), \"w\") as f:\n            f.write(\"data\")\n        return len(os.listdir(folder))\n    assert in_scratch(job) == 1 and not os.path.exists(seen[0])\n    def failing(folder):\n        seen.append(folder)\n        raise ValueError\n    try:\n        in_scratch(failing)\n        assert False\n    except ValueError:\n        pass\n    assert not os.path.exists(seen[-1])\n    assert scratch_file_lines([\"a\", \"b\"]) == [\"a\", \"b\"] and scratch_file_lines([]) == []")

### Exercise: Closing like a file

Write `Session(name)`: `closed` (a property), `query(text)` returning `f"{name}: {text}"` (or raising `ValueError("session is closed")` once closed), an **idempotent** `close()` that increments the class attribute `Session.closed_count` only the first time, and context manager support. Also `query_all(name, texts)`: run every query inside one session and return the results.

In [ ]:
class Session:
    closed_count = 0

    def __init__(self, name):
        ___

def query_all(name, texts):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_closing():\n    \"file-like conventions\"\n    Session.closed_count = 0\n    with Session(\"db\") as s:\n        assert s.query(\"SELECT 1\") == \"db: SELECT 1\" and not s.closed\n    assert s.closed and Session.closed_count == 1\n    s.close()\n    s.close()\n    assert Session.closed_count == 1\n    try:\n        s.query(\"again\")\n        assert False\n    except ValueError as e:\n        assert str(e) == \"session is closed\"\n    assert query_all(\"x\", [\"a\", \"b\"]) == [\"x: a\", \"x: b\"] and Session.closed_count == 2\n    try:\n        Session(\"p\").closed = True\n        assert False\n    except AttributeError:\n        pass")

### Exercise: A pool that always gets them back

Write `Pool(factory, size)`: makes `size` items up front. `lease()` is a context manager handing out a free item (raising `RuntimeError("pool exhausted")` if none) and always taking it back. If the body raises `ConnectionError`, the item is **broken**: discard it, make a fresh one with the factory, put that in the pool, and let the error carry on. `available()` and `made` (how many items the factory has made) report the state.

In [ ]:
from contextlib import contextmanager

class Pool:
    def __init__(self, factory, size):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_pool():\n    \"leases\"\n    counter = iter(range(100))\n    pool = Pool(lambda: f\"c{next(counter)}\", 2)\n    assert pool.available() == 2 and pool.made == 2\n    with pool.lease() as a:\n        assert pool.available() == 1\n        with pool.lease() as b:\n            assert a != b and pool.available() == 0\n            try:\n                with pool.lease():\n                    pass\n                assert False\n            except RuntimeError as e:\n                assert str(e) == \"pool exhausted\"\n    assert pool.available() == 2\n    try:\n        with pool.lease() as c:\n            raise ValueError\n    except ValueError:\n        pass\n    assert pool.available() == 2 and pool.made == 2\n    try:\n        with pool.lease() as d:\n            broken = d\n            raise ConnectionError\n    except ConnectionError:\n        pass\n    assert pool.available() == 2 and pool.made == 3\n    with pool.lease() as e1, pool.lease() as e2:\n        assert broken not in (e1, e2)")

### Exercise: A cooperative deadline

Write `Deadline(seconds, clock)`: a context manager whose `__enter__` notes the start time and returns the deadline; `check()` raises `TimeoutError(f"over by {extra:.1f}s")` if more than `seconds` have passed, and `remaining()` gives the seconds left (never below 0). Then `process(items, work, deadline_seconds, clock)`: call `work(item)` for each item inside a `Deadline`, checking it **before** each item, and return the list of results; on a timeout, the error propagates.

In [ ]:
class Deadline:
    def __init__(self, seconds, clock):
        ___

def process(items, work, deadline_seconds, clock):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_deadline():\n    \"checking as you go\"\n    now = [0.0]\n    clock = lambda: now[0]\n    def work(x):\n        now[0] += 1.0\n        return x * 2\n    assert process([1, 2, 3], work, 10, clock) == [2, 4, 6]\n    now[0] = 0.0\n    try:\n        process([1, 2, 3, 4, 5], work, 2.5, clock)\n        assert False\n    except TimeoutError as e:\n        assert str(e) == \"over by 0.5s\"\n    now[0] = 0.0\n    with Deadline(5, clock) as d:\n        now[0] = 3\n        assert d.remaining() == 2\n        now[0] = 9\n        assert d.remaining() == 0")

### Exercise: Counting what was opened

Write `Tracked(factory)`: wraps a factory of objects that have a `close()` method. `tracked.open(*args)` makes one with the factory, counts it as open, and returns an object whose `close()` (idempotent) also counts it closed; `tracked.leaks()` is how many are open now. Then `leaky_job(tracked, n, fail_at)` and `safe_job(tracked, n, fail_at)`: each opens n resources one by one and raises `RuntimeError` when about to open number `fail_at` (or never if `fail_at` is None); `leaky_job` closes them only at the end of normal success, `safe_job` with an `ExitStack` (using `contextlib.closing`) so nothing leaks either way.

In [ ]:
from contextlib import ExitStack, closing

class Tracked:
    def __init__(self, factory):
        ___

def leaky_job(tracked, n, fail_at):
    ___

def safe_job(tracked, n, fail_at):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("class Thing:\n    def close(self):\n        pass\ndef test_leaks():\n    \"counting\"\n    t = Tracked(Thing)\n    a = t.open()\n    b = t.open()\n    a.close()\n    a.close()\n    assert t.leaks() == 1\n    b.close()\n    assert t.leaks() == 0\n    t = Tracked(Thing)\n    leaky_job(t, 3, None)\n    assert t.leaks() == 0\n    try:\n        leaky_job(t, 5, 3)\n    except RuntimeError:\n        pass\n    assert t.leaks() == 3, \"the leaky version leaks on failure\"\n    t = Tracked(Thing)\n    for fail in (None, 0, 2):\n        try:\n            safe_job(t, 4, fail)\n        except RuntimeError:\n            pass\n    assert t.leaks() == 0")

### Bug hunt: A lease that's never returned

The item goes back to the pool only after a **successful** body, so an error loses it for good, and after one failure the pool reports `0` available. Return it in a `finally` so this prints `1`.

In [ ]:
from contextlib import contextmanager

free = ["conn"]

@contextmanager
def lease():
    item = free.pop()
    yield item
    free.append(item)

try:
    with lease():
        raise ConnectionResetError
except ConnectionResetError:
    pass
print(len(free))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"always return\"\n    expect_output(\"1\")")

### Bug hunt: Closing twice fails

This `close()` isn't idempotent: the second call (made by the `with` after an explicit close) tries to remove the session again and raises `ValueError`, printing `ValueError`. Make the second call do nothing, so this prints `[]`.

In [ ]:
open_sessions = []

class Session:
    def __init__(self):
        open_sessions.append(self)
    def close(self):
        open_sessions.remove(self)
    def __enter__(self):
        return self
    def __exit__(self, *exc):
        self.close()

try:
    with Session() as s:
        s.close()
    print(open_sessions)
except ValueError:
    print("ValueError")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"idempotent\"\n    expect_output(\"[]\")")

### Bug hunt: Releasing what was never acquired

The acquire is **inside** the `try`, so when acquiring fails (the pool is empty), the `finally` still "releases" an item that was never taken, and the pool ends up with an extra, made-up entry: it prints `2`. Acquire before the `try` so a failed acquire releases nothing and this prints `1`.

In [ ]:
free = ["only"]

def use(n):
    try:
        item = free.pop() if free else None
        if item is None:
            raise RuntimeError("empty")
        return item
    finally:
        free.append("only")

use(1)
try:
    free.clear()
    use(2)
except RuntimeError:
    pass
free.append("only")
print(len(free))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"acquire first\"\n    expect_output(\"1\")")

### Puzzle: A safe lease

Arrange the lines of a `@contextmanager` lease that always returns its item. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # True or False: __del__ is guaranteed to run as soon as an object is no longer used
q = ___   # True or False: a second call to an idempotent close() raises an error
r = ___   # the exception files raise when used after closing, as a string
s = ___   # True or False: a TemporaryDirectory's folder still exists after its with block
t = ___   # the warning category Python's development mode shows for files left open, as a string

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p is False and q is False and r == \"ValueError\", \"p, q and r\"\n    assert s is False and t == \"ResourceWarning\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a connection pool with health checks

Build a more realistic pool. Connections come from a factory and have `alive` (True until they break) and `close()`. Write `ConnectionPool(factory, size, max_uses=None)`:

- Start with no connections; create them **lazily**, up to `size` at once (counting both free and leased ones).
- `connection()`: a context manager that leases a connection: reuse a free one if possible (skipping and closing any that are no longer `alive`), otherwise make a new one if under `size`, otherwise raise `RuntimeError("pool exhausted")`.
- On return: if the body raised `ConnectionError`, or the connection isn't `alive`, or it has been used `max_uses` times, **close** it and don't keep it; otherwise put it back.
- `stats()`: a dict with `created`, `closed`, `free` and `leased` counts.
- `close_all()`: close every free connection; afterwards, leasing raises `RuntimeError("pool closed")`. Connections still leased are closed when they come back.

In [ ]:
from contextlib import contextmanager

class ConnectionPool:
    def __init__(self, factory, size, max_uses=None):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("class Conn:\n    made = 0\n    def __init__(self):\n        Conn.made += 1\n        self.id = Conn.made\n        self.alive = True\n        self.closed = False\n        self.uses = 0\n    def close(self):\n        self.closed = True\ndef test_pool():\n    \"a realistic pool\"\n    Conn.made = 0\n    pool = ConnectionPool(Conn, 2, max_uses=3)\n    assert pool.stats() == {\"created\": 0, \"closed\": 0, \"free\": 0, \"leased\": 0}\n    with pool.connection() as a:\n        with pool.connection() as b:\n            assert a is not b and pool.stats()[\"leased\"] == 2\n            try:\n                with pool.connection():\n                    pass\n                assert False\n            except RuntimeError as e:\n                assert str(e) == \"pool exhausted\"\n    assert pool.stats() == {\"created\": 2, \"closed\": 0, \"free\": 2, \"leased\": 0}\n    for _ in range(2):\n        with pool.connection() as c:\n            pass\n    assert pool.stats()[\"created\"] == 2\n    try:\n        with pool.connection() as d:\n            raise ConnectionError\n    except ConnectionError:\n        pass\n    assert d.closed and pool.stats()[\"closed\"] >= 1\n    with pool.connection() as e:\n        e.alive = False\n    assert e.closed\n    s = pool.stats()\n    assert s[\"free\"] + s[\"leased\"] <= 2\n    with pool.connection() as f:\n        pass\n    pool.close_all()\n    assert pool.stats()[\"free\"] == 0\n    try:\n        with pool.connection():\n            pass\n        assert False\n    except RuntimeError as err:\n        assert str(err) == \"pool closed\"")

## Warm-up

The temporary folder, and everything put inside it, is removed when the `with` ends. Nothing to forget.

## Not __del__

A class can define `__del__`, which runs when the object is about to be destroyed. It's tempting to close resources there, but **don't rely on it**:

- **When** it runs isn't guaranteed: CPython usually runs it as soon as the last reference goes, but objects in reference cycles wait for the garbage collector (next week), other Python implementations (PyPy) run it much later, and at interpreter exit it may not run at all.
- Exceptions raised in `__del__` are printed and **ignored**, not raised.
- It makes reasoning about a program depend on how memory happens to be managed.

Release resources **explicitly**: `with`, `close()`, `ExitStack`. If you need a safety net for objects someone forgot to close, `weakref.finalize` (Week 62) is more reliable than `__del__`, and Python's development mode (`python -X dev`) warns with a `ResourceWarning` when files are left open.

## Temporary space

The `tempfile` module makes scratch files and folders with unique names in the system's temporary area:

- `tempfile.TemporaryDirectory()`: a context manager giving the path of a new folder, deleted with its contents at the end.
- `tempfile.NamedTemporaryFile(mode="w", delete=True)`: an open file with a real path, deleted when closed. (On Windows, before Python 3.12, it can't be reopened by name while open; `delete_on_close=False` helps from 3.12.)
- `tempfile.TemporaryFile()`: an unnamed file for scratch data.
- `tempfile.mkdtemp()` and `mkstemp()`: you clean up yourself (combine with `ExitStack.callback`).

Tests (Week 24) use these constantly; pytest's `tmp_path` fixture is a temporary folder per test.

## Closing well

A resource class should follow the conventions of Python's own files:

- **`close()` is idempotent**: calling it twice is harmless (the second call does nothing). Cleanup code often can't know whether something was already closed.
- **Use after close is an error**: methods check a `closed` flag and raise `ValueError` (as files do), rather than failing mysteriously later.
- **A `closed` property** lets callers check.
- **A context manager** whose `__exit__` calls `close()`, so `with Resource() as r:` works.

## Pools

Some resources are expensive to create or limited in number: database connections, worker processes, licences. A **pool** creates a fixed number, hands them out, and takes them back for reuse. The danger is a resource that's taken and never returned (after an error), until the pool is empty and everything waits for ever. A context manager makes returning automatic: `with pool.lease() as conn: ...` returns `conn` to the pool in `__exit__`, whatever happened.

A good pool also decides what to do when it's empty (raise, or wait: waiting needs threads, Week 66), and what to do with a resource that **broke** while leased (throw it away and make a fresh one, rather than handing a broken connection to the next user).

## Deadlines

A long job (processing a big folder, retrying a flaky server) should stop after a time limit and still clean up. Python can't safely interrupt arbitrary code from outside (without threads or signals), so the usual approach is **cooperative**: a deadline object that the job **checks** between steps, raising `TimeoutError` when time is up. Because the job runs inside `with` statements, the error releases everything on its way out. Pass the clock in, as usual, so tests are exact.

## Testing for leaks

A test can check that code releases what it takes: wrap the factory or `open` so that every acquisition and release is counted, run the code (including failing cases), and assert that the counts match. Pools can report how many items are out; classes can expose `closed`. A leak caught in a test is much cheaper than one found when a server runs out of connections at 3 a.m.

## Practice

## Quiz

## Daily challenge

## Before you go

Look at a script you've written that opens files or makes temporary files: would it leave anything behind if it crashed half-way? Tomorrow is the Forge: a small job runner that puts this week's tools together.